# 01 Landing - Retail

**Audience:** participants learning the AIDP medallion pattern with PySpark.

**Prerequisites:** use the lab's shared compute and run the previous notebook first.

**Learning goal:** Loads the four workspace CSV files into the participant's Object Storage Landing prefixes.

## Outline

1. Inspect the participant-scoped inputs.
2. Transform and persist this medallion layer.
3. Register external tables when this layer owns them.
4. Verify the row counts printed by the final statements.


In [ ]:
import re
# oidlUtils is injected by AIDP Workbench; no import is required.

def required_parameter(name):
    value = oidlUtils.parameters.getParameter(name, "")
    if value is None or not str(value).strip():
        raise ValueError(f"Missing AIDP job parameter: {name}")
    return str(value).strip()

participant_key = required_parameter("participant_key")
lab_id = required_parameter("lab_id")
workspace_root = required_parameter("workspace_root")
bucket_name = required_parameter("bucket_name")
objectstorage_namespace = required_parameter("objectstorage_namespace")

if re.fullmatch(r"u_[0-9a-f]{16}", participant_key) is None:
    raise ValueError("Invalid participant_key")
if lab_id != 'retail':
    raise ValueError("This notebook belongs to a different lab")
if not workspace_root.startswith("/Workspace/medallon/"):
    raise ValueError("Invalid workspace_root")

from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql.types import StringType, StructField, StructType

source_root = Path(workspace_root) / "source"
specs = {'customers': {'filename': 'customers.csv', 'primary_key': ['customer_id'], 'foreign_keys': [], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'customer_id', 'type': 'STRING', 'required': True}, {'name': 'segment', 'type': 'STRING', 'required': True}, {'name': 'region', 'type': 'STRING', 'required': True}, {'name': 'loyalty_tier', 'type': 'STRING', 'required': True}, {'name': 'signup_date', 'type': 'DATE', 'required': True}, {'name': 'status', 'type': 'STRING', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}, 'products': {'filename': 'products.csv', 'primary_key': ['product_id'], 'foreign_keys': [], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'product_id', 'type': 'STRING', 'required': True}, {'name': 'category', 'type': 'STRING', 'required': True}, {'name': 'brand_label', 'type': 'STRING', 'required': True}, {'name': 'unit_cost', 'type': 'DOUBLE', 'required': True}, {'name': 'list_price', 'type': 'DOUBLE', 'required': True}, {'name': 'status', 'type': 'STRING', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}, 'orders': {'filename': 'orders.csv', 'primary_key': ['order_id'], 'foreign_keys': [['customer_id', 'customers', 'customer_id']], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'order_id', 'type': 'STRING', 'required': True}, {'name': 'customer_id', 'type': 'STRING', 'required': True}, {'name': 'order_time', 'type': 'TIMESTAMP', 'required': True}, {'name': 'channel', 'type': 'STRING', 'required': True}, {'name': 'region', 'type': 'STRING', 'required': True}, {'name': 'currency', 'type': 'STRING', 'required': True}, {'name': 'order_status', 'type': 'STRING', 'required': True}, {'name': 'discount_amount', 'type': 'DOUBLE', 'required': True}, {'name': 'declared_total', 'type': 'DOUBLE', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}, 'order_items': {'filename': 'order_items.csv', 'primary_key': ['order_id', 'line_number'], 'foreign_keys': [['order_id', 'orders', 'order_id'], ['product_id', 'products', 'product_id']], 'columns': [{'name': 'participant_key', 'type': 'STRING', 'required': True}, {'name': 'source_row_id', 'type': 'STRING', 'required': True}, {'name': 'order_id', 'type': 'STRING', 'required': True}, {'name': 'line_number', 'type': 'BIGINT', 'required': True}, {'name': 'product_id', 'type': 'STRING', 'required': True}, {'name': 'quantity', 'type': 'BIGINT', 'required': True}, {'name': 'unit_price', 'type': 'DOUBLE', 'required': True}, {'name': 'discount_amount', 'type': 'DOUBLE', 'required': True}, {'name': 'updated_at', 'type': 'TIMESTAMP', 'required': True}]}}
destinations = {"customers": f"oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/retail/customers/", "order_items": f"oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/retail/order_items/", "orders": f"oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/retail/orders/", "products": f"oci://{bucket_name}@{objectstorage_namespace}/01_landing/users/{participant_key}/retail/products/"}
source_files = sorted(source_root.glob("*.csv"))
assert {item.name for item in source_files} == {spec["filename"] for spec in specs.values()}

for dataset, spec in specs.items():
    source_file = source_root / spec["filename"]
    source_columns = [column for column in spec["columns"] if column["name"] != "participant_key"]
    raw_schema = StructType([StructField(column["name"], StringType(), True) for column in source_columns])
    frame = (spark.read.option("header", True).schema(raw_schema).csv(str(source_file))
        .withColumn("participant_key", F.lit(participant_key))
        .select(*[column["name"] for column in spec["columns"]]))
    source_count = frame.count()
    frame.write.mode("overwrite").option("header", True).csv(destinations[dataset])
    landing_count = spark.read.option("header", True).csv(destinations[dataset]).count()
    assert landing_count == source_count, f"Landing count mismatch for {dataset}"
    print(f"Landing {dataset}: {landing_count} rows")


## Expected result

Four CSV prefixes contain the same row totals as the source files.

**Exercise:** rerun this notebook and confirm that counts do not increase. All writes use
participant-exclusive paths and overwrite mode, so a second run is idempotent.

**Common pitfall:** do not replace the participant paths with shared locations. That would mix
different students' data. As an extension, query the registered tables with `spark.sql`.
